# 08 · Abstraer un patrón de diseño en una función de orden superior

> **En el examen:** es el **Ejercicio 4** típico (3-3,5 puntos) en casi todos los cursos
> desde 21-22. Siempre tiene la misma forma:
>
> *"Considérense las siguientes funciones `f1`, `f2`… basadas en un mismo patrón de diseño."*
> - **a)** Implementa una función de orden superior **`X`** que encapsule el patrón (≈ 1,5 p).
> - **b)** Reimplementa `f1`, `f2`… usando **`X`** (≈ 1,5 p).
>
> Es el ejercicio que más "asusta", pero tiene receta.

Requisitos: archivos 01, 04, 05 y 07. Aquí verás que `map`, `foldRight`, `foldLeft` y
`foldTree` **son exactamente esto**: patrones abstraídos.

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. La receta en 5 pasos

1. **Pon las funciones una al lado de la otra** y marca qué es **igual** en todas (la
   estructura: el `match`, la recursión, cómo se construye el resultado) y qué **cambia**.
2. **Lo que cambia se convierte en un parámetro.** Si lo que cambia es *un valor*, el
   parámetro es un valor; si es *un cálculo que depende de algo* (por ejemplo "lo que se le
   hace a la `i`"), el parámetro es **una función**.
3. **Generaliza los tipos.** Sustituye los tipos concretos (`Int`, `String`, `Boolean`…) por
   parámetros de tipo (`A`, `B`, `C`…). Ayuda: si en una función es `Int` y en otra
   `Boolean` en el mismo sitio, ese sitio es un tipo genérico.
4. **Escribe la HOF copiando la estructura común** y llamando a los parámetros-función donde
   antes había código concreto. Si es recursiva, la llamada recursiva pasa los mismos
   parámetros-función.
5. **Reimplementa cada función original** llamando a la HOF y pasando, como lambdas, los
   trozos que cambiaban.

**Consejo sobre la signatura:** pon la estructura de datos en una lista de parámetros y las
funciones en **otra** (`(t: ...)(f: ...)`). Así Scala deduce los tipos a partir de la
primera y no tienes que anotar las lambdas. El nombre de la HOF y a veces su signatura te
los da el enunciado: **úsala tal cual**.

---

## 2. Ejemplo guiado: `mapLeft` (25-26 Móstoles)

Funciones del enunciado (versión que corresponde con el texto):

In [ ]:
def add1(t: Either[Int, String]): Either[Int, String] =
    t match
        case Left(i)  => Left(i + 1)
        case Right(s) => Right(s)

def not(t: Either[Boolean, Char]): Either[Boolean, Char] =
    t match
        case Left(b)  => Left(!b)
        case Right(c) => Right(c)

def concat(s: String): Either[String, Float] => Either[String, Float] =
    case Left(s1) => Left(s1 + s)
    case Right(f) => Right(f)

**Paso 1 – comparar:**

| | `add1` | `not` | `concat` |
|---|---|---|---|
| estructura | `match`, `Left` → `Left(algo)`, `Right(x)` → `Right(x)` | igual | igual |
| lo que se hace al `Left` | `i + 1` | `!b` | `s1 + s` |
| tipo del Left | `Int` | `Boolean` | `String` |
| tipo del Right | `String` | `Char` | `Float` |

**Paso 2 – lo que cambia:** "lo que se le hace al valor del `Left`" → parámetro función
`f: A => C`.

**Paso 3 – tipos:** Left `A`, Right `B`, resultado de `f` es `C` (puede ser otro tipo).

**Paso 4 – la HOF:**

In [ ]:
def mapLeft[A, B, C](t: Either[A, B])(f: A => C): Either[C, B] =
    t match
        case Left(a)  => Left(f(a))
        case Right(b) => Right(b)

(La solución del profesor escribe `def mapLeft[A, B](t: Either[A, B])[C](f: A => C)`: en
Scala 3 se pueden poner parámetros de tipo **entre** listas de parámetros. Es lo mismo;
usa la forma que te resulte más clara.)

**Paso 5 – reimplementar:**

In [ ]:
def add1(e: Either[Int, String]): Either[Int, String] = mapLeft(e)(_ + 1)
def not(e: Either[Boolean, Char]): Either[Boolean, Char] = mapLeft(e)(!_)
def concat(s: String): Either[String, Float] => Either[String, Float] =
    e => mapLeft(e)(s1 => s1 + s)

### Variante con tuplas: `map_1` (25-26 Vicálvaro)

`add1(t) = (t._1 + 1, t._2)`, `not(t) = (!t._1, t._2)`… → se cambia el primer componente.

In [ ]:
def map_1[A, B, C](t: (A, B))(f: A => C): (C, B) =
    (f(t._1), t._2)

def add1(e: (Int, String)): (Int, String) = map_1(e)(_ + 1)
def not(e: (Boolean, Char)): (Boolean, Char) = map_1(e)(!_)
def concat(s: String): ((String, Float)) => (String, Float) =
    e => map_1(e)(_ + s)

> En el examen **Vicálvaro Bis** el enunciado pide `map_l` para **`Either`** (como
> `mapLeft`), pero la solución del archivo es la de tuplas. La correcta es la de `mapLeft`
> de arriba.

---

## 3. Las "familias" de patrones que salen

Casi todos los ejercicios 4 son una de estas familias. Reconocerla te da la mitad de la
solución.

| Familia | Idea | Ejemplos de examen |
|---|---|---|
| **map** | misma forma, se transforma el contenido | `mapLeft`, `map_1`, `map` de árboles, `lift1st` |
| **fold** | se consume la estructura y se calcula un resultado | `foldRight`, `foldLeft`, `foldTree`, `fold` (Leaf/Branch), `paramorphism`, `foldLeftUntil`, `dyv` |
| **unfold** | se **genera** una estructura a partir de una semilla | `unfold`, `repeat`, `unfoldTree` |
| **traverse** | como `map`, pero la función puede fallar (`Option`/`Either`); si falla una, falla todo | `traverse` en listas y en árboles, `sequence` |

---

## 4. Ejemplos resueltos de exámenes

### Familia traverse: `traverse` sobre listas con `Option` (22-23 Vicálvaro)

`parse` convierte textos a números (falla si alguno no es número), `validate` comprueba que
todos sean ≥ 0. Las dos tienen esta forma:

In [ ]:
def parse(l: List[String]): Option[List[Int]] =
    l match
        case Nil => Some(Nil)
        case head :: tail =>
            (head.toIntOption, parse(tail)) match      // ← lo único que cambia es head.toIntOption
                case (Some(h), Some(t)) => Some(h :: t)
                case _ => None

Lo que cambia es "qué se le hace a la cabeza" (una función `A => Option[B]`):

In [ ]:
def traverse[A, B](l: List[A])(f: A => Option[B]): Option[List[B]] =
    l match
        case Nil => Some(Nil)
        case head :: tail =>
            (f(head), traverse(tail)(f)) match
                case (Some(h), Some(t)) => Some(h :: t)
                case _ => None

def parse(l: List[String]): Option[List[Int]] = traverse(l)(_.toIntOption)
def validate(l: List[Int]): Option[List[Int]] = traverse(l)(n => if n < 0 then None else Some(n))
def ints(l: List[Double]): Option[List[Int]] =
    traverse(l)(d => if !d.isValidInt then None else Some(d.toInt))

La versión con `Either` (22-23 Móstoles) es igual, pero devolviendo el **primer** error:

In [ ]:
def traverse[A, B, E](l: List[A])(f: A => Either[E, B]): Either[E, List[B]] =
    l match
        case Nil => Right(Nil)
        case head :: tail =>
            (f(head), traverse(tail)(f)) match
                case (Right(h), Right(t)) => Right(h :: t)
                case (Right(_), Left(e))  => Left(e)
                case (Left(e), _)         => Left(e)      // el de la izquierda tiene prioridad

### Familia traverse en árboles (23-24 Vicálvaro)

In [ ]:
def traverse[A, B](t: Tree[A])(f: A => Option[B]): Option[Tree[B]] =
    t match
        case Empty() => Some(Empty())
        case Node(left, a, right) =>
            (traverse(left)(f), f(a), traverse(right)(f)) match
                case (Some(l), Some(b), Some(r)) => Some(Node(l, b, r))
                case _ => None

// heads: el primer elemento de cada lista del árbol (None si alguna está vacía)
def heads[A](t: Tree[List[A]]): Option[Tree[A]] =
    traverse(t):
        case Nil => None
        case h :: _ => Some(h)

// divide: a / cada nodo (None si algún nodo es 0)
def divide(a: Int)(t: Tree[Int]): Option[Tree[Int]] =
    traverse(t)(b => if b == 0 then None else Some(a / b))

// apartado c): traverse = map + sequence
def traverse2[A, B](t: Tree[A])(f: A => Option[B]): Option[Tree[B]] =
    sequence(map(t)(f))

### Familia fold: `foldLeftUntil` (20-21 Ordinaria)

Un `foldLeft` que **se para antes** si el acumulado cumple una condición:

In [ ]:
def foldLeftUntil[A, B](l: List[A])(acc: B)(cond: B => Boolean)(f: (B, A) => B): B =
    l match
        case h :: t if !cond(acc) => foldLeftUntil(t)(f(acc, h))(cond)(f)
        case _ => acc      // lista vacía o condición cumplida: paro

// buscar el valor de una clave (el primero)
def lookup[A, B](list: List[(A, B)])(a: A): Option[B] =
    foldLeftUntil(list)(Option.empty[B])(_.isDefined):   // paro en cuanto lo encuentro
        case (_, (`a`, b)) => Some(b)
        case (acc, _) => acc

### Familia fold: `paramorphism` (23-24 Móstoles)

Como `foldRight` pero la función recibe **también** el resto de la lista:

In [ ]:
def paramorphism[A, B](list: List[A])(nil: B)(cons: (A, List[A], B) => B): B =
    list match
        case Nil => nil
        case h :: tail => cons(h, tail, paramorphism(tail)(nil)(cons))

def tails[A](l: List[A]): List[List[A]] =
    paramorphism(l)(List(List[A]()))((h, tail, tailSol) => (h :: tail) :: tailSol)

### Familia fold: divide y vencerás genérico `dyv` (21-22 Vicálvaro)

Un problema `P` es o atómico (`Left`) o se parte en dos (`Right((p1, p2))`):

In [ ]:
def dyv[P, S](problem: P)(
        decompose: P => Either[P, (P, P)],
        atomic: P => S,
        compose: (S, S) => S): S =
    decompose(problem) match
        case Left(base) => atomic(base)
        case Right((p1, p2)) =>
            compose(dyv(p1)(decompose, atomic, compose), dyv(p2)(decompose, atomic, compose))

def mergeSort(numbers: Array[Int]): Array[Int] =
    def split(ns: Array[Int]): Either[Array[Int], (Array[Int], Array[Int])] =
        if ns.length <= 1 then Left(ns)
        else Right((ns.slice(0, ns.length / 2), ns.slice(ns.length / 2, ns.length)))
    dyv(numbers)(split, ns => ns, merge)      // merge viene dado en el enunciado

### Familia unfold: `unfold` (23-24 Móstoles)

`zip` y `range` **generan** una lista a partir de una semilla. En cada paso: o paro
(`None`), o produzco un elemento y una semilla nueva (`Some((elemento, nuevaSemilla))`).

In [ ]:
def unfold[A, B](next: B => Option[(A, B)])(seed: B): List[A] =
    next(seed) match
        case None => Nil
        case Some((a, b)) => a :: unfold(next)(b)

def zip[A, B](pair: (List[A], List[B])): List[(A, B)] =
    unfold[(A, B), (List[A], List[B])] {
        case (h1 :: t1, h2 :: t2) => Some(((h1, h2), (t1, t2)))
        case _ => None
    }(pair)

def range(max: Int)(min: Int): List[Int] =
    unfold((m: Int) => if m <= max then Some((m, m + 1)) else None)(min)

(En `zip` la lambda de `case` va en la primera lista de parámetros y la semilla `(pair)` en
la segunda; por eso se anotan los tipos `[(A, B), (List[A], List[B])]`.)

### Familia unfold: `repeat` (24-25 Vicálvaro)

In [ ]:
def genFib(max: Int): List[Int] =
    def step(out: List[Int], next: (Int, Int)): List[Int] =
        if next._1 > max then out
        else step(next._1 :: out, (next._2, next._1 + next._2))
    step(Nil, (0, 1)).reverse

Lo que cambia entre `genFib` y `genCollatz`: el estado inicial, cómo se pasa al siguiente
estado produciendo un elemento, y cuándo parar →

In [ ]:
def repeat[A, B](from: A)(f: A => (B, A))(until: A => Boolean): List[B] =
    def step(out: List[B], next: A): List[B] =
        if until(next) then out
        else
            val (b, next2) = f(next)
            step(b :: out, next2)
    step(Nil, from).reverse

def genFib(max: Int): List[Int] =
    repeat(from = (0, 1)) { case (n2, n1) => (n2, (n1, n1 + n2)) }(until = _._1 > max)

def genCollatz(n: Int): List[Int] =
    repeat(from = n)(m => (m, if m % 2 == 0 then m / 2 else 3 * m + 1))(until = _ == 1) :+ 1

def grouped[A](l: List[A], size: Int): List[List[A]] =
    repeat(from = l)(_.splitAt(size))(until = _ == Nil)

### Familia unfold en árboles: `unfoldTree` (24-25 Móstoles)

In [ ]:
def unfoldTree[A, B](seed: B)(f: B => Option[(B, A, B)]): Tree[A] =
    f(seed) match
        case None => Empty()
        case Some((b1, a, b2)) => Node(unfoldTree(b1)(f), a, unfoldTree(b2)(f))

// árbol cuyo recorrido en preorden es la lista
def genTree[A](l: List[A]): Tree[A] =
    unfoldTree(l):
        case Nil => None
        case head :: tail =>
            val (half1, half2) = tail.splitAt(tail.length / 2)
            Some((half1, head, half2))

// árbol sintáctico de una expresión
enum Expr:
    case Add(x: Expr, y: Expr)
    case Mul(x: Expr, y: Expr)
    case Lit(i: Int)
import Expr._

def genAST(e: Expr): Tree[Either[Int, Char]] =
    unfoldTree(Option(e)):                  // semilla Option: None = "aquí no hay hijo"
        case None => None
        case Some(Lit(i)) => Some((None, Left(i), None))
        case Some(Add(e1, e2)) => Some((Some(e1), Right('+'), Some(e2)))
        case Some(Mul(e1, e2)) => Some((Some(e1), Right('*'), Some(e2)))

Truco de `genAST`: un literal no tiene hijos, pero `unfoldTree` exige dar **dos semillas
hijas**. Por eso la semilla es un `Option[Expr]`: `None` significa "árbol vacío".

---

## 5. Errores típicos

- **No usar el nombre/signatura que da el enunciado.** Si dice `repeat(from)(f)(until)`,
  respeta ese orden de parámetros.
- **Olvidar pasar los parámetros-función en la llamada recursiva** (`traverse(tail)(f)`, no
  `traverse(tail)`).
- **Dejar tipos concretos en la HOF** (`Int` donde debería ir `A`).
- **En el apartado b) volver a escribir la recursión.** La gracia es que cada función sea
  una línea que llama a la HOF.

---

## 6. Ejercicios para practicar

### De exámenes reales

| HOF | Funciones que la usan | Examen |
|---|---|---|
| `mapLeft` | `add1`, `not`, `concat` | 25-26 Móstoles (y Bis) |
| `map_1` | `add1`, `not`, `concat` | 25-26 Vicálvaro |
| `unfoldTree` | `genAST`, `genTree` | 24-25 Móstoles |
| `repeat` | `genFib`, `genCollatz`, `grouped` | 24-25 Vicálvaro |
| `unfold` | `zip`, `range` | 23-24 Móstoles |
| `paramorphism` | `tails` | 23-24 Móstoles |
| `traverse` (árbol) | `heads`, `divide` | 23-24 Vicálvaro |
| `traverse` (Either) | `parse`, `validate`, `ints` | 22-23 Móstoles |
| `traverse` (Option) | `parse`, `validate`, `ints` | 22-23 Vicálvaro |
| `fold` | `isEven`, `total` | 22-23 Vicálvaro Especial |
| `map` / `foldTree` (árbol) | `duplicate`, `fusion`… | 22-23 Extraordinaria |
| `dyv` | `mergeSort` | 21-22 Vicálvaro |
| `foldLeftUntil` | `lookup` | 20-21 Ordinaria |

### Ejercicios nuevos

1. Abstrae el patrón de estas dos funciones en una HOF `mapRight` y reimplementa ambas:
   ```scala
   def dobleR(e: Either[String, Int]): Either[String, Int] =
       e match { case Left(s) => Left(s); case Right(n) => Right(n * 2) }
   def longR(e: Either[Boolean, String]): Either[Boolean, Int] =
       e match { case Left(b) => Left(b); case Right(s) => Right(s.length) }
   ```
2. Abstrae en `mapOption` y reimplementa:
   ```scala
   def inc(o: Option[Int]): Option[Int] = o match { case None => None; case Some(n) => Some(n + 1) }
   def len(o: Option[String]): Option[Int] = o match { case None => None; case Some(s) => Some(s.length) }
   ```
3. Usa `unfold` para escribir `countdown(n: Int): List[Int]` (`countdown(3) == List(3, 2, 1)`).
4. Usa `unfold` para escribir `digits(n: Int): List[Int]` con los dígitos al revés
   (`digits(123) == List(3, 2, 1)`; `digits(0) == Nil`).

<details>
<summary>Soluciones</summary>

```scala
// 1
def mapRight[A, B, C](e: Either[A, B])(f: B => C): Either[A, C] =
    e match
        case Left(a) => Left(a)
        case Right(b) => Right(f(b))
def dobleR(e: Either[String, Int]): Either[String, Int] = mapRight(e)(_ * 2)
def longR(e: Either[Boolean, String]): Either[Boolean, Int] = mapRight(e)(_.length)

// 2
def mapOption[A, B](o: Option[A])(f: A => B): Option[B] =
    o match
        case None => None
        case Some(a) => Some(f(a))
def inc(o: Option[Int]): Option[Int] = mapOption(o)(_ + 1)
def len(o: Option[String]): Option[Int] = mapOption(o)(_.length)

// 3
def countdown(n: Int): List[Int] =
    unfold((k: Int) => if k <= 0 then None else Some((k, k - 1)))(n)

// 4
def digits(n: Int): List[Int] =
    unfold((k: Int) => if k == 0 then None else Some((k % 10, k / 10)))(n)
```

</details>

Siguiente: **99-chuleta.ipynb** (resumen de una página para repasar antes del examen).